In [ ]:
import os
from google.cloud import documentai

# TODO(developer): Set these variables before running the script.
PROJECT_ID = "future-oasis-254818"
LOCATION = "us"  # Processor location, e.g., "us" or "eu"
PROCESSOR_ID = "5f225445c9546be8"  # The ID of your Document OCR processor
FILE_PATH = "/Users/siyuliang/Documents/UW/ocr/data/test/Jarring_Prov_2_005_v.Lanczos.800.65.jpg"
MIME_TYPE = "image/jpeg"  # Mime type of the input file

# Optional: Define language hints based on our previous discussion
# Relevant codes: "ug" (Uyghur), "uz" (Uzbek), "fa" (Persian), "ar" (Arabic)
# Experiment with different combinations
# LANGUAGE_HINTS = ["ug", "uz", "fa", "ar"]
LANGUAGE_HINTS = ["ug", "uz", "fa", "ar"]
# LANGUAGE_HINTS = ["ug"]  # Example: ["ug", "uz", "fa", "ar"]

def process_document_ocr_sample(
    project_id: str, location: str, processor_id: str, file_path: str, mime_type: str, language_hints: list[str] | None = None
):
    """
    Processes a document using the Document OCR processor with language hints.
    """

    # You must set the `api_endpoint` if you use a location other than "us".
    opts = {"api_endpoint": f"{location}-documentai.googleapis.com"}

    # Instantiates a client
    client = documentai.DocumentProcessorServiceClient(client_options=opts)

    # The full resource name of the processor, e.g.:
    # projects/project-id/locations/location/processors/processor-id
    name = client.processor_path(project_id, location, processor_id)

    # Read the file into memory
    try:
        with open(file_path, "rb") as image:
            image_content = image.read()
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return
    except Exception as e:
        print(f"Error reading file: {e}")
        return

    # Load Binary Data into Document AI RawDocument Structure
    raw_document = documentai.RawDocument(content=image_content, mime_type=mime_type)

    # Configure the process request
    # Include ProcessOptions with OcrConfig and language hints if provided
    process_options = None
    if language_hints:
        ocr_config = documentai.OcrConfig(
            hints=documentai.OcrConfig.Hints(language_hints=language_hints)
        )
        process_options = documentai.ProcessOptions(ocr_config=ocr_config)
        print(f"Processing with language hints: {language_hints}")
    else:
         print("Processing without specific language hints.")


    request = documentai.ProcessRequest(
        name=name,
        raw_document=raw_document,
        process_options=process_options, # Include options here
        # Skip human review for the synchronous processing request.
        skip_human_review=True,
    )

    try:
        # Use the Document AI client to process the sample form
        result = client.process_document(request=request)
    except Exception as e:
        print(f"Error during Document AI processing: {e}")
        return

    # For a full list of Document object attributes, please reference this page:
    # https://cloud.google.com/document-ai/docs/reference/rest/v1/Document
    document = result.document

    # Read the full text of the document
    print("\nFull document text:")
    print(document.text)

# --- Run the sample ---
if __name__ == "__main__":
    if not PROJECT_ID or PROJECT_ID == "YOUR_PROJECT_ID":
        print("Please set the PROJECT_ID variable in the script.")
    elif not os.path.exists(FILE_PATH):
         print(f"Error: The file path '{FILE_PATH}' does not exist.")
    else:
        process_document_ocr_sample(
            project_id=PROJECT_ID,
            location=LOCATION,
            processor_id=PROCESSOR_ID,
            file_path=FILE_PATH,
            mime_type=MIME_TYPE,
            language_hints=LANGUAGE_HINTS # Pass hints here
        )

Processing with language hints: ['ug']

Full document text:
A
دورلار رساله که عمل قبلی در لار و نیند
کا سیب روغن کش ایک قیلیب دور لار رسالہ
ساقلاماب دور لار رساله که عمل فیلماب
دورلار رساله اینجیلا کی پر مرید لارکہ
مذکور شریعت و طریقت و حقیقت و معرفت
و مذہب و ملت و فرض واجب سنت و
مستجد وادب و آرکان شرایط لارینی بقدر
چال بیلکایلار نامه نیام بای بجای بیلمان
روغن کشی ایک قبلغان بولسه لار جنرای آخرت
را تقویت که کرفتار بولغایلار بور سال بینی
ہفتہ
317
3



In [6]:
!pip install --upgrade google-cloud-documentai jiwer

  Attempting uninstall: jiwer
    Found existing installation: jiwer 3.0.5
    Uninstalling jiwer-3.0.5:
      Successfully uninstalled jiwer-3.0.5
  Attempting uninstall: google-cloud-documentai
    Found existing installation: google-cloud-documentai 3.4.0
    Uninstalling google-cloud-documentai-3.4.0:
      Successfully uninstalled google-cloud-documentai-3.4.0


In [14]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Processes documents listed in a JSON split file using Document AI OCR,
calculates average WER and CER against the ground truth, and saves
detailed results to a TSV file.

Assumes the following directory structure:
project_root/
├─ data/jarring_manuscripts_data/  <- JSON splits, images folder, and TSV output live here
└─ src/evaluate_ocr.py             <- run this script from here
"""

import json
import os
import sys
import time
import unicodedata # For potential normalization
import re
import csv # <-- Import csv module
from pathlib import Path
from google.cloud import documentai
import jiwer # For WER/CER calculation

# --- Configuration ---

# --- Google Cloud Configuration ---
# TODO(developer): Set these variables before running the script.
PROJECT_ID = "future-oasis-254818"  # Your Google Cloud project ID (e.g., chaghatayocr-project-123)
LOCATION = "us"  # Processor location (e.g., "us" or "eu")
PROCESSOR_ID = "5f225445c9546be8" # The ID of your Document OCR processor (e.g., 5f225445c9546be8)

# --- Data File Configuration ---
# Select which split file to process (e.g., train_split.json, val_split.json, test_split.json)
SPLIT_FILENAME = "val_split.json"
# Define the output TSV filename
OUTPUT_FILENAME = f"ocr_evaluation_results_{Path(SPLIT_FILENAME).stem}.tsv" # e.g., ocr_evaluation_results_train_split.tsv

# --- OCR Configuration ---
# Optional: Define language hints
LANGUAGE_HINTS = ["ug", "uz", "fa", "ar"]

# Limit the number of documents to process for quick testing (-1 for all)
MAX_DOCS_TO_PROCESS = -1 # Set to a small number like 5 for testing

# --- End Configuration ---


# --- Resolve Directories ---
try:
    # Assumes the script is in project_root/src/
    SRC_DIR = Path(__file__).resolve().parent
except NameError:
     # Fallback for interactive sessions (e.g., Jupyter, IPython)
     # Assumes interactive session is started from project_root/src/
    SRC_DIR = Path.cwd()

# Data directory is expected one level up and then into data/jarring_manuscripts_data
DATA_DIR = SRC_DIR.parent / "data" / "jarring_manuscripts_data"
SPLIT_FILE_PATH = DATA_DIR / SPLIT_FILENAME
OUTPUT_FILE_PATH = DATA_DIR / OUTPUT_FILENAME # <-- Define output path
PROJECT_ROOT = SRC_DIR.parent

if not SPLIT_FILE_PATH.exists():
     relative_data_dir = DATA_DIR.relative_to(PROJECT_ROOT) if PROJECT_ROOT in DATA_DIR.parents else DATA_DIR
     print(f"Error: Split file '{SPLIT_FILENAME}' not found in {relative_data_dir}")
     sys.exit(1)

print(f"Using split file: {SPLIT_FILE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Expecting images relative to project root: {PROJECT_ROOT}")
print(f"Results will be saved to: {OUTPUT_FILE_PATH.relative_to(PROJECT_ROOT)}")


# --- Helper Functions ---
# (process_document_ocr, get_ground_truth_text, normalize_text, calculate_cer, calculate_wer remain the same)
def process_document_ocr(
    client: documentai.DocumentProcessorServiceClient,
    processor_name: str,
    file_path: Path,
    mime_type: str,
    language_hints: list[str] | None = None
) -> str | None:
    """Processes a single document using the Document AI client."""
    if not file_path.is_file():
        print(f"  Error: Image file not found at resolved path: {file_path}")
        return None
    try:
        with open(file_path, "rb") as image:
            image_content = image.read()
    except Exception as e:
        print(f"  Error reading image file {file_path}: {e}")
        return None
    raw_document = documentai.RawDocument(content=image_content, mime_type=mime_type)
    process_options = None
    if language_hints:
        ocr_config = documentai.OcrConfig(hints=documentai.OcrConfig.Hints(language_hints=language_hints))
        process_options = documentai.ProcessOptions(ocr_config=ocr_config)
    request = documentai.ProcessRequest(name=processor_name, raw_document=raw_document, process_options=process_options, skip_human_review=True)
    try:
        result = client.process_document(request=request)
        return result.document.text
    except Exception as e:
        print(f"  Error during Document AI processing for {file_path.name}: {e}")
        return None

def get_ground_truth_text(record: dict) -> str:
    """Extracts and concatenates ground truth lines from a record."""
    lines_text = [line.get("arabic_text", "") for line in record.get("lines", [])]
    return " ".join(lines_text).strip()

def normalize_text(text: str) -> str:
    """Basic text normalization for WER/CER calculation."""
    if not text: return ""
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def calculate_cer(ground_truth: str, hypothesis: str) -> float:
    """Calculates Character Error Rate."""
    if not ground_truth and not hypothesis: return 0.0
    if not ground_truth: return 1.0
    if not hypothesis: return 1.0
    try: return jiwer.cer(ground_truth, hypothesis)
    except Exception as e: print(f"    Error calculating CER: {e}"); return 1.0

def calculate_wer(ground_truth: str, hypothesis: str) -> float:
    """Calculates Word Error Rate."""
    if not ground_truth and not hypothesis: return 0.0
    if not ground_truth: return 1.0
    if not hypothesis: return 1.0
    try: return jiwer.wer(ground_truth, hypothesis)
    except Exception as e: print(f"    Error calculating WER: {e}"); return 1.0

# --- Main Execution ---
def main():
    # Basic validation of config
    if not PROJECT_ID or PROJECT_ID == "YOUR_PROJECT_ID":
        print("Error: Please set the PROJECT_ID variable in the script.")
        sys.exit(1)
    if not PROCESSOR_ID or PROCESSOR_ID == "YOUR_PROCESSOR_ID":
        print("Error: Please set the PROCESSOR_ID variable in the script.")
        sys.exit(1)

    # Load the specified split data
    try:
        with open(SPLIT_FILE_PATH, 'r', encoding='utf-8') as f:
            split_data = json.load(f)
        print(f"Loaded {len(split_data)} records from {SPLIT_FILENAME}")
    except Exception as e:
        print(f"Error loading {SPLIT_FILENAME}: {e}")
        sys.exit(1)

    # Initialize Document AI Client
    try:
        opts = {"api_endpoint": f"{LOCATION}-documentai.googleapis.com"}
        client = documentai.DocumentProcessorServiceClient(client_options=opts)
        processor_name = client.processor_path(PROJECT_ID, LOCATION, PROCESSOR_ID)
        print(f"Using Processor: {processor_name}")
        if LANGUAGE_HINTS:
            print(f"Using Language Hints: {LANGUAGE_HINTS}")
    except Exception as e:
        print(f"Error initializing Document AI client: {e}")
        print("Ensure you have authenticated (e.g., `gcloud auth application-default login`)")
        sys.exit(1)

    # --- Setup CSV/TSV Output ---
    try:
        # Open file for writing - use utf-8 encoding and handle newlines correctly
        with open(OUTPUT_FILE_PATH, 'w', newline='', encoding='utf-8') as tsvfile:
            # Use csv.writer with tab delimiter for TSV
            writer = csv.writer(tsvfile, delimiter='\t')

            # Write Header Row
            header = [
                "Manuscript_ID", "Surface_ID", "Image_Path_Relative",
                "WER", "CER", "Ground_Truth_Normalized", "OCR_Output_Normalized"
            ]
            writer.writerow(header)

            # --- Process documents and write results ---
            results_metrics = [] # Still collect metrics for average calculation
            processed_count = 0

            for i, record in enumerate(split_data):
                if MAX_DOCS_TO_PROCESS != -1 and processed_count >= MAX_DOCS_TO_PROCESS:
                    print(f"\nReached MAX_DOCS_TO_PROCESS limit ({MAX_DOCS_TO_PROCESS}). Stopping.")
                    break

                manuscript_id = record.get("manuscript_id", "Unknown")
                surface_id = record.get("surface_id", "Unknown")
                relative_path_from_json = record.get("absolute_image_path") # Path relative to PROJECT_ROOT/data

                print(f"\nProcessing record {i+1}/{len(split_data)}: {manuscript_id} / {surface_id}...")
                # print(f"  Relative path from JSON: {relative_path_from_json}") # Optional debug

                if not relative_path_from_json:
                    print("  Skipping record: Missing 'absolute_image_path' key in JSON record.")
                    continue

                # Construct full absolute path
                absolute_image_path = PROJECT_ROOT / "data" / relative_path_from_json
                # print(f"  Attempting to access image at: {absolute_image_path}") # Optional debug

                # Basic MIME type inference
                file_suffix = absolute_image_path.suffix.lower()
                if file_suffix in [".jpg", ".jpeg"]: mime_type = "image/jpeg"
                elif file_suffix == ".png": mime_type = "image/png"
                elif file_suffix in [".tif", ".tiff"]: mime_type = "image/tiff"
                elif file_suffix == ".pdf": mime_type = "application/pdf"
                else:
                    print(f"  Skipping record: Unknown MIME type for {absolute_image_path.name} (suffix: {file_suffix}). Add handling if needed.")
                    continue

                # Get OCR result (Hypothesis)
                start_time = time.time()
                ocr_text = process_document_ocr(client, processor_name, absolute_image_path, mime_type, LANGUAGE_HINTS)
                end_time = time.time()
                print(f"  OCR completed in {end_time - start_time:.2f} seconds.")

                if ocr_text is None:
                    print("  Skipping metrics calculation and TSV row due to file/OCR error.")
                    continue

                # Get Ground Truth
                gt_text = get_ground_truth_text(record)

                # Normalize both texts before comparison
                normalized_gt = normalize_text(gt_text)
                normalized_ocr = normalize_text(ocr_text)

                # Calculate WER and CER
                page_wer = calculate_wer(normalized_gt, normalized_ocr)
                page_cer = calculate_cer(normalized_gt, normalized_ocr)

                print(f"  WER: {page_wer:.4f}")
                print(f"  CER: {page_cer:.4f}")

                # --- Write Row to TSV ---
                row_data = [
                    manuscript_id,
                    surface_id,
                    relative_path_from_json, # Store the relative path used
                    f"{page_wer:.4f}", # Format metrics as strings if desired
                    f"{page_cer:.4f}",
                    normalized_gt,
                    normalized_ocr
                ]
                writer.writerow(row_data)
                # --- End Write Row ---

                results_metrics.append({"wer": page_wer, "cer": page_cer})
                processed_count += 1

    except IOError as e:
        print(f"\nError opening or writing to output file {OUTPUT_FILE_PATH}: {e}")
        sys.exit(1)
    except Exception as e:
        print(f"\nAn unexpected error occurred during processing or writing: {e}")
        sys.exit(1)


    # --- Calculate and Print Average Metrics ---
    if results_metrics:
        avg_wer = sum(r["wer"] for r in results_metrics) / len(results_metrics)
        avg_cer = sum(r["cer"] for r in results_metrics) / len(results_metrics)
        print("\n--- Average Metrics (Printed to Console) ---")
        print(f"Successfully processed and evaluated {len(results_metrics)} documents.")
        print(f"Average WER: {avg_wer:.4f}")
        print(f"Average CER: {avg_cer:.4f}")
        print("--------------------------------------------")
        print(f"Detailed results saved to: {OUTPUT_FILE_PATH.relative_to(PROJECT_ROOT)}")
    else:
        print("\nNo documents were successfully processed to calculate average metrics.")

if __name__ == "__main__":
    main()

Using split file: data/jarring_manuscripts_data/val_split.json
Expecting images relative to project root: /Users/siyuliang/Documents/UW/ocr
Results will be saved to: data/jarring_manuscripts_data/ocr_evaluation_results_val_split.tsv
Loaded 59 records from val_split.json
Using Processor: projects/future-oasis-254818/locations/us/processors/5f225445c9546be8
Using Language Hints: ['ug', 'uz', 'fa', 'ar']

Processing record 1/59: Jarring_Prov_8 / 1a...
  OCR completed in 1.40 seconds.
  WER: 0.3333
  CER: 0.1000

Processing record 2/59: Jarring_Prov_8 / 1b...
  OCR completed in 1.26 seconds.
  WER: 0.6106
  CER: 0.1713

Processing record 3/59: Jarring_Prov_8 / 2a...
  OCR completed in 1.33 seconds.
  WER: 0.8584
  CER: 0.2765

Processing record 4/59: Jarring_Prov_8 / 2b...
  OCR completed in 1.23 seconds.
  WER: 0.6102
  CER: 0.1569

Processing record 5/59: Jarring_Prov_8 / 3a...
  OCR completed in 1.44 seconds.
  WER: 0.7583
  CER: 0.1937

Processing record 6/59: Jarring_Prov_8 / 3b...
  

In [19]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Performs error analysis on OCR evaluation results stored in TSV files.
Generates summary statistics and plots, handling outliers for summaries/plots,
and adjusting per-manuscript plot axes.

Assumes the following directory structure:
project_root/
├─ data/jarring_manuscripts_data/  <- TSV input files live here, plots saved here
└─ src/analyze_ocr_errors.py       <- run this script from here
"""

import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import jiwer
from pathlib import Path
from collections import Counter
import numpy as np

# --- Configuration ---
TOP_N_ERRORS = 15
CER_PLOT_THRESHOLD = 1.5
WER_PLOT_THRESHOLD = 3.0 # Increased threshold slightly based on previous output if needed

# --- Resolve Directories ---
try:
    SRC_DIR = Path(__file__).resolve().parent
except NameError:
    SRC_DIR = Path.cwd()

DATA_DIR = SRC_DIR.parent / "data" / "jarring_manuscripts_data"
PLOT_DIR = DATA_DIR
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# --- Helper Functions ---
# (load_data, plot_distributions, analyze_worst_pages, analyze_common_errors remain the same
#  as in the previous version)

def load_data(data_directory: Path) -> pd.DataFrame | None:
    """Loads and concatenates data from train, val, test TSV files."""
    all_data = []
    splits = ["train", "val", "test"]
    found_files = 0
    for split in splits:
        filename = f"ocr_evaluation_results_{split}_split.tsv"
        file_path = data_directory / filename
        if file_path.exists():
            try:
                df_split = pd.read_csv(file_path, sep='\t')
                df_split['WER'] = pd.to_numeric(df_split['WER'], errors='coerce')
                df_split['CER'] = pd.to_numeric(df_split['CER'], errors='coerce')
                df_split.dropna(subset=['WER', 'CER'], inplace=True)
                df_split['Ground_Truth_Normalized'] = df_split['Ground_Truth_Normalized'].fillna('').astype(str)
                df_split['OCR_Output_Normalized'] = df_split['OCR_Output_Normalized'].fillna('').astype(str)
                df_split['split'] = split
                all_data.append(df_split)
                print(f"Loaded {len(df_split)} records from {filename}")
                found_files += 1
            except Exception as e: print(f"Error loading or processing {filename}: {e}")
        else: print(f"Warning: File not found - {filename}")
    if not all_data:
        print(f"Error: No TSV result files found in {data_directory}. Cannot perform analysis.")
        return None
    df_combined = pd.concat(all_data, ignore_index=True)
    print(f"\nTotal records loaded across {found_files} file(s): {len(df_combined)}")
    df_combined['WER'] = df_combined['WER'].astype(float)
    df_combined['CER'] = df_combined['CER'].astype(float)
    return df_combined

def plot_distributions(df: pd.DataFrame, wer_threshold: float, cer_threshold: float, plot_dir: Path):
    """Generates histograms and box plots for WER and CER, filtering outliers."""
    print(f"\nGenerating distribution plots (filtering WER > {wer_threshold}, CER > {cer_threshold} for plots)...")
    df_filtered = df[(df['WER'] <= wer_threshold) & (df['CER'] <= cer_threshold)].copy()
    if len(df_filtered) < len(df): print(f"  Filtered out {len(df) - len(df_filtered)} records as outliers for distribution plots/averages.")
    if df_filtered.empty: print("  No data remaining after filtering outliers. Skipping distribution plots."); return

    print("\n--- Average Performance Metrics (Outliers Removed for this Calculation) ---")
    print(df_filtered.groupby('split')[['WER', 'CER']].agg(['mean', 'median', 'std']))
    print(f"--- (Based on WER <= {wer_threshold}, CER <= {cer_threshold}) ---")

    plt.style.use('seaborn-v0_8-darkgrid')
    # WER Plots
    plt.figure(figsize=(12, 6)); sns.histplot(data=df_filtered, x='WER', kde=True, bins=30, hue='split'); plt.title(f'Distribution of Word Error Rate (WER <= {wer_threshold})'); plt.xlabel('WER'); plt.ylabel('Frequency'); plt.tight_layout(); plt.savefig(plot_dir / "wer_distribution_filtered.png"); plt.close()
    plt.figure(figsize=(10, 6)); sns.boxplot(data=df_filtered, x='split', y='WER', order=['train', 'val', 'test']); plt.title(f'Word Error Rate (WER <= {wer_threshold}) by Data Split'); plt.xlabel('Split'); plt.ylabel('WER'); plt.ylim(bottom=0); plt.tight_layout(); plt.savefig(plot_dir / "wer_boxplot_filtered.png"); plt.close()
    # CER Plots
    plt.figure(figsize=(12, 6)); sns.histplot(data=df_filtered, x='CER', kde=True, bins=30, hue='split'); plt.title(f'Distribution of Character Error Rate (CER <= {cer_threshold})'); plt.xlabel('CER'); plt.ylabel('Frequency'); plt.tight_layout(); plt.savefig(plot_dir / "cer_distribution_filtered.png"); plt.close()
    plt.figure(figsize=(10, 6)); sns.boxplot(data=df_filtered, x='split', y='CER', order=['train', 'val', 'test']); plt.title(f'Character Error Rate (CER <= {cer_threshold}) by Data Split'); plt.xlabel('Split'); plt.ylabel('CER'); plt.ylim(bottom=0); plt.tight_layout(); plt.savefig(plot_dir / "cer_boxplot_filtered.png"); plt.close()
    print(f"Filtered distribution plots saved to: {plot_dir}")

def analyze_worst_pages(df: pd.DataFrame, top_n: int):
    """Identifies and prints pages with the highest WER and CER."""
    print(f"\n--- Top {top_n} Pages with Highest CER (Worst First) ---")
    df_worst_cer = df.sort_values(by='CER', ascending=False).head(top_n)
    print(df_worst_cer[['Manuscript_ID', 'Surface_ID', 'CER', 'WER']].to_string(index=False, float_format="%.4f"))
    print(f"\n--- Top {top_n} Pages with Highest WER (Worst First) ---")
    df_worst_wer = df.sort_values(by='WER', ascending=False).head(top_n)
    print(df_worst_wer[['Manuscript_ID', 'Surface_ID', 'WER', 'CER']].to_string(index=False, float_format="%.4f"))
    print("----------------------------------------------------")

# --- MODIFIED FUNCTION ---
def analyze_by_manuscript(df: pd.DataFrame, plot_dir: Path):
    """Calculates and plots average WER/CER per manuscript, adjusting axis start."""
    print("\nAnalyzing performance by manuscript...")
    if 'Manuscript_ID' not in df.columns:
        print("  Skipping analysis: 'Manuscript_ID' column not found.")
        return
    if df[['WER', 'CER']].isnull().any().any():
        print("  Warning: Found NaN values in WER/CER, excluding them from manuscript averages.")
        df_perf = df.dropna(subset=['WER', 'CER'])
    else:
        df_perf = df

    if df_perf.empty:
        print("  No valid data for manuscript performance analysis.")
        return

    manuscript_performance = df_perf.groupby('Manuscript_ID')[['WER', 'CER']].mean().reset_index()

    print("\n--- Average Performance per Manuscript (Raw Averages) ---")
    print(manuscript_performance.sort_values(by='CER', ascending=False).to_string(index=False, float_format="%.4f"))
    print("--------------------------------------------------------")

    # Find min values for axis adjustment (use a small buffer)
    min_avg_wer = manuscript_performance['WER'].min() if not manuscript_performance.empty else 0
    min_avg_cer = manuscript_performance['CER'].min() if not manuscript_performance.empty else 0
    # Add a small buffer, ensuring it doesn't go below 0
    wer_start_point = max(0, min_avg_wer * 0.95)
    cer_start_point = max(0, min_avg_cer * 0.95)

    # --- Plotting ---
    plt.style.use('seaborn-v0_8-darkgrid')

    # CER per Manuscript (Adjusted Axis)
    plt.figure(figsize=(12, max(6, 0.5 * len(manuscript_performance))))
    manuscript_performance_cer = manuscript_performance.sort_values(by='CER', ascending=False)
    ax_cer = sns.barplot(data=manuscript_performance_cer, y='Manuscript_ID', x='CER', hue='Manuscript_ID', palette='viridis', legend=False)
    plt.title('Average Character Error Rate (CER) by Manuscript (Adjusted Axis)')
    plt.xlabel('Average CER')
    plt.ylabel('Manuscript ID')
    # Set the left x-limit to start near the minimum observed average CER
    ax_cer.set_xlim(left=cer_start_point)
    plt.tight_layout()
    plt.savefig(plot_dir / "cer_by_manuscript_adjusted.png")
    plt.close()

    # WER per Manuscript (Adjusted Axis)
    plt.figure(figsize=(12, max(6, 0.5 * len(manuscript_performance))))
    manuscript_performance_wer = manuscript_performance.sort_values(by='WER', ascending=False)
    ax_wer = sns.barplot(data=manuscript_performance_wer, y='Manuscript_ID', x='WER', hue='Manuscript_ID', palette='viridis', legend=False)
    plt.title('Average Word Error Rate (WER) by Manuscript (Adjusted Axis)')
    plt.xlabel('Average WER')
    plt.ylabel('Manuscript ID')
    # Set the left x-limit to start near the minimum observed average WER
    ax_wer.set_xlim(left=wer_start_point)
    plt.tight_layout()
    plt.savefig(plot_dir / "wer_by_manuscript_adjusted.png")
    plt.close()

    print(f"Per-manuscript plots (adjusted axes) saved to: {plot_dir}")
# --- END MODIFIED FUNCTION ---


def analyze_common_errors(df: pd.DataFrame):
    """Analyzes total character substitutions, deletions, insertions."""
    print(f"\n--- Analyzing Total Character Errors ---")
    if 'Ground_Truth_Normalized' not in df.columns or 'OCR_Output_Normalized' not in df.columns: print("  Skipping analysis: Required text columns not found."); return
    df = df.dropna(subset=['Ground_Truth_Normalized', 'OCR_Output_Normalized'])
    df['Ground_Truth_Normalized'] = df['Ground_Truth_Normalized'].astype(str)
    df['OCR_Output_Normalized'] = df['OCR_Output_Normalized'].astype(str)

    total_subs, total_dels, total_ins, total_hits, total_gt_chars = 0, 0, 0, 0, 0
    for index, row in df.iterrows():
        gt, hyp = row['Ground_Truth_Normalized'], row['OCR_Output_Normalized']
        total_gt_chars += len(gt)
        if not gt or not hyp:
            if not gt and hyp: total_ins += len(hyp)
            elif gt and not hyp: total_dels += len(gt)
            continue
        try:
            output = jiwer.process_characters(gt, hyp)
            total_subs += output.substitutions; total_dels += output.deletions; total_ins += output.insertions; total_hits += output.hits
        except Exception as e: print(f"  Warning: Error processing alignment for record index {index}: {e}"); continue

    total_errors = total_subs + total_dels + total_ins
    calculated_cer = total_errors / total_gt_chars if total_gt_chars > 0 else 0
    print(f"\nTotal Characters (Ground Truth): {total_gt_chars}")
    print(f"Total Correct Characters (Hits): {total_hits}")
    print(f"Total Character Substitutions:   {total_subs}")
    print(f"Total Character Deletions:       {total_dels}")
    print(f"Total Character Insertions:      {total_ins}")
    print(f"Total Errors (S+D+I):            {total_errors}")
    print(f"Overall CER (calculated from errors): {calculated_cer:.4f}")
    print("-----------------------------------------")


# --- Main Execution ---
def main():
    print("--- Starting OCR Error Analysis ---")
    df_results = load_data(DATA_DIR)
    if df_results is None or df_results.empty: print("Exiting due to missing or empty data."); sys.exit(1)

    plot_distributions(df_results, WER_PLOT_THRESHOLD, CER_PLOT_THRESHOLD, PLOT_DIR)
    analyze_worst_pages(df_results, TOP_N_ERRORS)
    analyze_by_manuscript(df_results, PLOT_DIR) # Calls the modified function
    analyze_common_errors(df_results)
    print("\n--- Analysis Complete ---")

if __name__ == "__main__":
    main()

--- Starting OCR Error Analysis ---
Loaded 152 records from ocr_evaluation_results_train_split.tsv
Loaded 59 records from ocr_evaluation_results_val_split.tsv
Loaded 53 records from ocr_evaluation_results_test_split.tsv

Total records loaded across 3 file(s): 264

Generating distribution plots (filtering WER > 3.0, CER > 1.5 for plots)...
  Filtered out 4 records as outliers for distribution plots/averages.

--- Average Performance Metrics (Outliers Removed for this Calculation) ---
            WER                         CER                  
           mean  median       std      mean  median       std
split                                                        
test   0.931830  0.9586  0.107752  0.283504  0.2861  0.046203
train  0.869441  0.8710  0.182425  0.289052  0.2768  0.111007
val    0.778048  0.7884  0.113609  0.240102  0.2288  0.064928
--- (Based on WER <= 3.0, CER <= 1.5) ---
Filtered distribution plots saved to: /Users/siyuliang/Documents/UW/ocr/data/jarring_manuscripts_d